In [2]:
import numpy as np 
import pandas as pd
from pathlib import Path 
from tqdm.auto import tqdm

# In this notebook

* ### Create stimuli for cross-condition model tests (e.g. harmonic target w inharmonic distractor)
* ### Format stimuli for compat with online participant experiments 

___
### Create stimuli for cross-condition tests (e.g. harmonic target w inharmonic distractor)

Cuerrently we have pandas dataframes that include the stimuli waveforms:
* `/om2/user/imgriff/datasets/timit/harmonic_timit/all_targets_harmonic_single_distractor_0dB_SNR_jitter_fn_render.pdpkl` 
* `/om2/user/imgriff/datasets/timit/whispered_timit/all_targets_whispered_single_distractor_0dB_SNR.pdpkl`
* `/om2/user/imgriff/datasets/timit/inharmonic_timit/all_targets_inharmonic_single_distractor_0dB_SNR.pdpkl`

Plan: 
* Save cue, target, and distractor signals to directories 
* Will need to adapt pytorch dataloader to enable reading from dirs and mixing signals on the fly


In [3]:
path_to_timit_dir = Path("/om2/user/imgriff/datasets/timit/")

harm_df_name = "/om2/user/imgriff/datasets/timit/harmonic_timit/all_targets_harmonic_single_distractor_0dB_SNR_jitter_fn_render.pdpkl" 
whispered_df_name = "/om2/user/imgriff/datasets/timit/whispered_timit/all_targets_whispered_single_distractor_0dB_SNR.pdpkl"
inharm_df_name = "/om2/user/imgriff/datasets/timit/inharmonic_timit/all_targets_inharmonic_single_distractor_0dB_SNR.pdpkl"

## Load in dfs 

harm_df = pd.read_pickle(harm_df_name)
inharm_df = pd.read_pickle(inharm_df_name)
whispered_df = pd.read_pickle(whispered_df_name)

In [4]:
## Make sure distractor ixs match 
(inharm_df._original_distractor_timit_indices.values == harm_df._original_distractor_timit_indices.values).all()

True

In [5]:
## Make sure distractor ixs match 
(whispered_df._original_distractor_timit_indices.values == harm_df._original_distractor_timit_indices.values).all()

True

#### Will need to combine target & distractor signals from each df

Define helper functions 

In [6]:
def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

def make_mixture(target, distractor, snr):
    mixture, _ = combine_with_noise(target, distractor, snr) # first_scale_factor unused
    mixture, mixture_scale_factor = rms_normalize(mixture)
    return mixture, mixture_scale_factor

In [9]:
## Specify combinations 

signal_combs = {'harmonic_target_inharmonic_distractor': {'target_df':harm_df, 'dist_df':inharm_df },
                'inharmonic_target_harmonic_distractor': {'target_df':inharm_df, 'dist_df':harm_df },
                'harmonic_target_whispered_distractor': {'target_df':harm_df, 'dist_df':whispered_df },
                'whispered_target_harmonic_distractor': {'target_df':whispered_df, 'dist_df':harm_df },
                'whispered_target_inharmonic_distractor': {'target_df':whispered_df, 'dist_df':inharm_df },
                'inharmonic_target_whispered_distractor': {'target_df':inharm_df, 'dist_df':whispered_df },
}

In [13]:
inharm_df.columns

Index(['signal', 'speaker', 'speaker_sex', 'sentence_type', 'word_int',
       'mixture_signal', 'cue_signal', 'cue_speaker', 'cue_word',
       'distractor_signal', '_original_distractor_timit_indices',
       '_original_cue_timit_index', 'distractor_words', 'distractor_speakers',
       'distractor_conditions', 'distractor_sex', 'snrs', 'word',
       '_original_timit_index', 'source', 'sr', 'signal_length', 'sentence_id',
       'dialect_region', 'data_split', 'distractor_word_ints'],
      dtype='object')

In [10]:
## Loop through and generate mixtures
SNR = 0 

for comb_name, cond_df_dict in tqdm(signal_combs.items(), leave=True, desc=f'Combination'):
    
    target_df = cond_df_dict['target_df']
    dist_df = cond_df_dict['dist_df']

    out_path  = path_to_timit_dir / comb_name 
    out_path.mkdir(exist_ok=True, parents=True)
    out_name = out_path / f"{comb_name}_0dB_SNR.pdpkl"

    # use data from target_df, just need distractor signal from dist_df
    new_df = target_df.copy()
    dist_signals = dist_df.distractor_signal.values
    target_signals = target_df.signal.values
    new_signals = [make_mixture(target, dist, SNR)[0] for target, dist in tqdm(zip(target_signals, dist_signals), total=len(target_signals), leave=False)]
    new_df['mixture_signal'] = new_signals
    # break
    new_df.to_pickle(out_name)


Combination:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

  0%|          | 0/902 [00:00<?, ?it/s]

In [11]:
out_path

PosixPath('/om2/user/imgriff/datasets/timit/inharmonic_target_whispered_distractor')

In [27]:
import IPython.display as ipd 

In [28]:
ipd.Audio(new_df.mixture_signal[0], rate=20_000)